In [3]:
import os
from google.colab import files
import pandas as pd

# Check and upload the CSV if necessary.
if not os.path.exists("btc_trading_data.csv"):
    print("File 'btc_trading_data.csv' not found. Please upload it now.")
    uploaded = files.upload()  # Upload your CSV file.

# Now load the CSV.
df = pd.read_csv("btc_trading_data.csv")
print(df.head())


File 'btc_trading_data.csv' not found. Please upload it now.


Saving btc_trading_data.csv to btc_trading_data.csv
                        Date        Open        High         Low       Close  \
0  2014-09-17 00:00:00+00:00  465.864014  468.174011  452.421997  457.334015   
1  2014-09-18 00:00:00+00:00  456.859985  456.859985  413.104004  424.440002   
2  2014-09-19 00:00:00+00:00  424.102997  427.834991  384.532013  394.795990   
3  2014-09-20 00:00:00+00:00  394.673004  423.295990  389.882996  408.903992   
4  2014-09-21 00:00:00+00:00  408.084991  412.425995  393.181000  398.821014   

     Volume  Dividends  Stock Splits  
0  21056800        0.0           0.0  
1  34483200        0.0           0.0  
2  37919700        0.0           0.0  
3  36863600        0.0           0.0  
4  26580100        0.0           0.0  


In [5]:
import os
import time
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap, pmap, grad, random
from jax.experimental import pjit
from jax.sharding import Mesh, PositionalSharding
from functools import partial

# ========================================
# Step 0: Ensure Trading CSV is Uploaded in Colab
# ========================================
from google.colab import files
if not os.path.exists("btc_trading_data.csv"):
    print("File 'btc_trading_data.csv' not found. Please upload it now.")
    uploaded = files.upload()

# ========================================
# Pipeline Configuration (Small Scale for Trading)
# ========================================
PIPE_MAX_RECURSION_DEPTH    = 100_000      # Maximum recursion depth for pipeline
PIPE_TOTAL_DEPTH            = 100_000      # Total recursion depth (full fusion)
PIPE_OPTIMAL_DEPTH_STEP     = PIPE_TOTAL_DEPTH  # Full fusion in one call
PIPE_DIMENSIONAL_CONSTRAINT = 0.8
MAX_PIPE_BATCH_SIZE         = 1_000_000    # Use up to 1M samples for prototyping

PIPE_VAL_CLAMP_LOW  = -100.0
PIPE_VAL_CLAMP_HIGH =  100.0

# Use up to 8 devices (or available devices)
PIPE_NUM_DEVICES = min(8, jax.device_count())
devices = jax.devices()[:PIPE_NUM_DEVICES]
mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

DATA_LAKE_DIR = "datalake"
if not os.path.exists(DATA_LAKE_DIR):
    os.makedirs(DATA_LAKE_DIR)

# -------------------------------
# Pipeline Functions
# -------------------------------
@jit
def pipe_dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / ((scale_factor * 20) + 1)) * PIPE_DIMENSIONAL_CONSTRAINT

@jit
def pipe_stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

@partial(jit, static_argnames=["depth"])
def pipe_dppu_with_dynamic_pi_phi(x, depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = pipe_stabilize_depth(jnp.minimum(depth, PIPE_MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = pipe_dynamic_pi(i, scale_factor)
        phi_dyn = pipe_dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * PIPE_DIMENSIONAL_CONSTRAINT
        safe_val = jnp.clip(val, PIPE_VAL_CLAMP_LOW, PIPE_VAL_CLAMP_HIGH)
        new_val = jnp.sin(safe_val * scale * pi_dyn) * jnp.exp(-safe_val / (phi_dyn + 10))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

def pipe_branch_recycle(x, num_branches=2, branch_depth=PIPE_OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    xs = jnp.stack([x] * num_branches, axis=0)
    branch_fn = vmap(lambda xi: pipe_dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.sum(branch_outputs, axis=0)

# Use pjit with positional arguments (no kwargs for static parameters)
@partial(pjit.pjit,
         in_shardings=(sharding,),
         out_shardings=sharding,
         static_argnames=("num_branches", "branch_depth", "scale_factor"))
def pipe_process_full(x, num_branches, branch_depth, scale_factor):
    return pipe_branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)

def run_pipeline(raw_data):
    # Limit to MAX_PIPE_BATCH_SIZE samples
    data = raw_data[:MAX_PIPE_BATCH_SIZE]
    sharded_input = jax.device_put(data, sharding)
    start_time = time.time()
    # Pass positional arguments
    final_output = pipe_process_full(sharded_input, 2, PIPE_OPTIMAL_DEPTH_STEP, 1.0)
    final_output = jax.device_get(final_output)
    jax.block_until_ready(final_output)
    elapsed = time.time() - start_time
    mean_val = float(jnp.mean(final_output))
    return final_output, elapsed, mean_val

def save_to_data_lake(data, filename="processed_trading_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    np.save(filepath, np.array(data))
    print(f"Saved processed data to {filepath}")

def load_from_data_lake(filename="processed_trading_data.npy"):
    filepath = os.path.join(DATA_LAKE_DIR, filename)
    if os.path.exists(filepath):
        data = np.load(filepath)
        print(f"Loaded processed data from {filepath}")
        return data
    else:
        print(f"No file found at {filepath}")
        return None

# ========================================
# Trading Data Ingestion & Preprocessing
# ========================================
def load_trading_data(csv_path, column="Close"):
    # Load trading data from CSV using pandas.
    df = pd.read_csv(csv_path)
    data = df[column].values.astype(np.float32)
    # Normalize using z-score normalization.
    data = (data - np.mean(data)) / np.std(data)
    return data

# ========================================
# RNN Configuration & Functions (for Trading)
# ========================================
input_dim = 1
hidden_dim = 32
output_dim = 1
seq_length = 100  # Each sequence will be 100 samples

def get_num_sequences(data_length, seq_length):
    return data_length // seq_length

def init_rnn_params(key, input_dim, hidden_dim, output_dim):
    k1, k2, k3, k4 = random.split(key, 4)
    W_xh = random.normal(k1, (input_dim, hidden_dim)) * 0.1
    W_hh = random.normal(k2, (hidden_dim, hidden_dim)) * 0.1
    b_h = jnp.zeros((hidden_dim,))
    W_hy = random.normal(k3, (hidden_dim, output_dim)) * 0.1
    b_y = jnp.zeros((output_dim,))
    return {"W_xh": W_xh, "W_hh": W_hh, "b_h": b_h, "W_hy": W_hy, "b_y": b_y}

def rnn_step(params, h, x):
    h_next = jnp.tanh(jnp.dot(x, params["W_xh"]) + jnp.dot(h, params["W_hh"]) + params["b_h"])
    return h_next

def rnn_forward(params, inputs):
    # inputs: (seq_length, input_dim)
    def step_fn(h, x):
        h_new = rnn_step(params, h, x)
        return h_new, h_new
    h0 = jnp.zeros((hidden_dim,))
    final_h, _ = lax.scan(step_fn, h0, inputs)
    output = jnp.dot(final_h, params["W_hy"]) + params["b_y"]
    return output

def rnn_loss_fn(params, batch_inputs, batch_targets):
    def single_loss(inputs, target):
        pred = rnn_forward(params, inputs)
        return jnp.mean((pred - target) ** 2)
    losses = jax.vmap(single_loss)(batch_inputs, batch_targets)
    return jnp.mean(losses)

@jit
def rnn_train_step(params, batch_inputs, batch_targets, learning_rate=0.001):
    grads = grad(rnn_loss_fn)(params, batch_inputs, batch_targets)
    new_params = {k: params[k] - learning_rate * grads[k] for k in params}
    loss = rnn_loss_fn(params, batch_inputs, batch_targets)
    return new_params, loss

# ========================================
# Main: End-to-End Trading DRNN System
# ========================================
if __name__ == "__main__":
    # ----- Step 1: Trading Data Ingestion -----
    trading_csv = "btc_trading_data.csv"  # Ensure this CSV is uploaded in Colab.
    print("Loading trading data...")
    raw_trading_data = load_trading_data(trading_csv, column="Close")
    print(f"Loaded trading data of length: {raw_trading_data.shape[0]}")

    # ----- Step 2: Run Pipeline on Trading Data -----
    print("Running processing pipeline on trading data...")
    processed_data, pipe_elapsed, pipe_mean = run_pipeline(jnp.array(raw_trading_data))
    print(f"Pipeline completed in {pipe_elapsed:.2f} sec with mean output {pipe_mean:.6f}")
    save_to_data_lake(processed_data, filename="processed_trading_data.npy")

    # ----- Step 3: Prepare Data for RNN -----
    processed_data_np = np.array(processed_data)
    total_length = processed_data_np.shape[0]
    num_sequences = get_num_sequences(total_length, seq_length)
    # Reshape into (num_sequences, seq_length, 1)
    rnn_inputs = processed_data_np[:num_sequences * seq_length].reshape((num_sequences, seq_length, 1))
    # For trading, we'll use the last value in each sequence as the target.
    rnn_targets = rnn_inputs[:, -1, :]  # shape: (num_sequences, 1)
    print(f"RNN training data: {num_sequences} sequences of length {seq_length}")

    # ----- Step 4: Train the RNN on Trading Data -----
    key = random.PRNGKey(0)
    rnn_params = init_rnn_params(key, input_dim, hidden_dim, output_dim)
    num_epochs = 5
    batch_size = 32
    num_batches = num_sequences // batch_size

    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for i in range(num_batches):
            batch_inputs = rnn_inputs[i*batch_size:(i+1)*batch_size]
            batch_targets = rnn_targets[i*batch_size:(i+1)*batch_size]
            rnn_params, loss_val = rnn_train_step(rnn_params, batch_inputs, batch_targets)
            epoch_loss += loss_val
        epoch_loss /= num_batches
        print(f"Epoch {epoch+1}, Loss: {epoch_loss:.6f}")

    # ----- Step 5: Test the RNN -----
    sample_input = rnn_inputs[0]
    sample_pred = rnn_forward(rnn_params, sample_input)
    print("Sample RNN prediction:", sample_pred)
    print("Sample RNN target:", rnn_targets[0])


Loading trading data...
Loaded trading data of length: 3814
Running processing pipeline on trading data...
Pipeline completed in 6.60 sec with mean output -0.021164
Saved processed data to datalake/processed_trading_data.npy
RNN training data: 38 sequences of length 100
Epoch 1, Loss: 2.216234
Epoch 2, Loss: 2.203506
Epoch 3, Loss: 2.190849
Epoch 4, Loss: 2.178263
Epoch 5, Loss: 2.165745
Sample RNN prediction: [-0.01336701]
Sample RNN target: [-2.0435092]
